<a href="https://colab.research.google.com/github/Rapsim/IPEO_DeepL_group15/blob/raph/explo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### code adapté de l'exo 9

In [1]:
import os
import torch
from torch.utils.data import dataset
from torch.utils.data import DataLoader
import numpy as np
import tifffile


In [2]:
# Number of classes: 0 = nodata to 37 = last class in CSV ?
NUM_CLASSES = 38


In [3]:
dataroot = "sample"

### images mean and std 

In [5]:
def compute_s2_mean_std(s2_dir, file_prefix="train"):
    s2_files = sorted(
        f for f in os.listdir(s2_dir)
        if f.startswith(file_prefix) and f.endswith(".tif")
    )
    
    sum_band = None
    sumsq_band = None
    total_pixels = 0

    for i, fname in enumerate(s2_files):
        path = os.path.join(s2_dir, fname)
        img = tifffile.imread(path).astype(np.float32)

        # ensure (C,H,W)
        if img.ndim == 2:
            img = img[np.newaxis, ...]
        elif img.ndim == 3 and img.shape[-1] in (12, 13):
            img = np.transpose(img, (2, 0, 1))  # (H,W,C)->(C,H,W)

        img = img / 10000.0  # scale to approx [0,1]
        C, H, W = img.shape

        if sum_band is None:
            sum_band = np.zeros(C, dtype=np.float64)
            sumsq_band = np.zeros(C, dtype=np.float64)

        flat = img.reshape(C, -1)
        sum_band += flat.sum(axis=1)
        sumsq_band += (flat**2).sum(axis=1)
        total_pixels += H * W

    mean = sum_band / total_pixels
    var = sumsq_band / total_pixels - mean**2
    std = np.sqrt(np.maximum(var, 1e-12))

    return mean, std

# Example: use on small training sample S2 folder 
# s2_dir_for_stats = "/content/drive/MyDrive/IPEO_DeepL_data/sample/S2" 
s2_dir_for_stats = "sample/S2" 
S2_MEAN, S2_STD = compute_s2_mean_std(s2_dir_for_stats, file_prefix="train")
print("S2_MEAN:", S2_MEAN)
print("S2_STD:", S2_STD)


S2_MEAN: [0.1139377  0.0890742  0.08113043 0.06680196 0.09717837 0.19016036
 0.23556764 0.2318273  0.27211989 0.05309604 0.18734722 0.08958261]
S2_STD: [0.00712784 0.01266245 0.01809968 0.03474503 0.03448118 0.03047756
 0.0388264  0.04093171 0.04471077 0.01083777 0.08132065 0.05494784]


### dataset


In [6]:


class VaihingenDataset(dataset.Dataset):
    """
    Custom Dataset class that loads Sentinel-2 image patches and
    land cover masks from your Amazon dataset.

    We keep the old name 'VaihingenDataset' so the rest of the
    notebook works without big changes.
    """

    def __init__(self, data_root, split="train"):
        """
        data_root: folder that contains the subfolders:
            data_root/S2/*.tif
            data_root/labels/*.tif

        split: "train", "val", or "test"
        """
        super().__init__()
        self.data_root = data_root
        self.split = split

        self.s2_dir = os.path.join(data_root, "S2")
        self.label_dir = os.path.join(data_root, "labels")

        # all filenames in S2 folder
        all_fnames = sorted(
            [f for f in os.listdir(self.s2_dir) if f.endswith(".tif")]
        )

        # separate train and test by prefix
        train_fnames = [f for f in all_fnames if f.startswith("train")]
        test_fnames  = [f for f in all_fnames if f.startswith("test")]

        if split in ("train", "val"):
            # deterministic 80/20 split of the train_ files
            n_train = int(0.8 * len(train_fnames))
            train_fnames = sorted(train_fnames)
            if split == "train":
                selected = train_fnames[:n_train]
            else:  # "val"
                selected = train_fnames[n_train:]
        elif split == "test":
            selected = sorted(test_fnames)
        else:
            raise ValueError(f"Unknown split: {split}")

        self.fnames = selected

        print(f"{split} split has {len(self.fnames)} samples.")

    def __len__(self):
        return len(self.fnames)

    def _read_tiff_image(self, path):
        """
        Read a .tif file as a numpy array and return it as (C, H, W).

        Works whether data is stored as (H, W, C) or (C, H, W) or (H, W).
        """
        arr = tifffile.imread(path)  # numpy array

        if arr.ndim == 2:
            # single band -> (1, H, W)
            arr = arr[np.newaxis, ...]
        elif arr.ndim == 3:
            # guess if channels are first or last
            if arr.shape[0] in (1, 3, 4, 12, 64):
                # assume (C, H, W)
                pass
            elif arr.shape[-1] in (1, 3, 4, 12, 64):
                # assume (H, W, C)
                arr = np.transpose(arr, (2, 0, 1))
            else:
                raise ValueError(f"Unexpected image shape {arr.shape} for {path}")
        else:
            raise ValueError(f"Unexpected ndim {arr.ndim} for {path}")

        return arr

    def __getitem__(self, idx):
        fname = self.fnames[idx]

        # Sentinel-2 image path and label path
        img_path = os.path.join(self.s2_dir, fname)
        label_path = os.path.join(self.label_dir, fname)

        # ---- Read Sentinel-2 patch ----
        img = self._read_tiff_image(img_path).astype(np.float32)  # (C, H, W)

        # Sentinel-2 typical values up to ~10000 -> scale to ~[0, 1]
        img = img / 10000.0

        img = torch.from_numpy(img)  # (C,H,W)

        if self.input_type == "S2":
            # apply per-band normalization using precomputed mean/std
            img = (img - self.S2_MEAN[:, None, None]) / self.S2_STD[:, None, None]

        # ---- Read label mask ----
        mask = tifffile.imread(label_path)  # often (H, W) or (1, H, W)

        if mask.ndim == 3:
            # squeeze if there is a singleton channel dimension
            if mask.shape[0] == 1:
                mask = mask[0]
            elif mask.shape[-1] == 1:
                mask = mask[..., 0]
            else:
                raise ValueError(f"Unexpected mask shape {mask.shape} for {label_path}")

        if mask.ndim != 2:
            raise ValueError(f"Mask must be 2D, got shape {mask.shape}")

        mask = mask.astype(np.int64)  # required for CrossEntropyLoss

        # ---- Convert to torch tensors ----
        inputs = torch.from_numpy(img)    # (C, H, W), C should be 12
        labels = torch.from_numpy(mask)   # (H, W), ints in [0..62] (0 = nodata)

        return inputs, labels


def load_dataloader(batch_size, split="train"):
    """
    split in {"train", "val", "test"}.

    The global variable `data_root` should point to the folder that
    contains the S2/ and labels/ folders.
    """
    return DataLoader(
        VaihingenDataset(data_root, split=split),
        batch_size=batch_size,
        shuffle=(split == "train"),
        num_workers=2,
    )


### visualise some images

In [ ]:
import os
%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

#discrete color scheme
cMap = ListedColormap(['grey', 'darkgreen', 'lawngreen', 'red', 'orange', 'black'])     #  #'Buildings', 'Tree', 'Low Vegetation', 'Clutter', 'Car', 'Impervious'

dataset_train = VaihingenDataset(data_root)

# draw a random sample
idx = torch.randint(0, len(dataset_train), (1,))
data, target = dataset_train.__getitem__(idx)
print(f'Image tensor size: {data.size()}')
print(f'Label tensor size: {target.size()}')

# visualise
plt.figure()
plt.imshow(data[:3,...].permute(1,2,0).numpy())     # first three bands: NIR-R-G
plt.title('Input: NIR-R-G satellite imagery')
plt.show()
fig = plt.figure()